<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook begins an SV summary table comparing short-read GATK-SV and long-read joint SV callsets. It downloads the VCFs, lists overlapping samples (including the ~1,027 non-HPRC long-read samples), and tabulates ALT alleles and SVTYPEs.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas as pd
import numpy as np
import pysam
import os

In [ ]:
import pysam
from pysam import VariantFile

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']
workspace = os.environ['WORKSPACE_NAME']
namespace = os.environ['WORKSPACE_NAMESPACE']

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Short-read SV callset (GATK-SV Phase I joint VCF), downloaded once, and its sample IDs.
if not os.path.exists("AoU_srWGS_SV_PhaseI.vcf.gz"):
    !gsutil -m cp {AOU_LR_DATA}/aou-wgs-sv/phase1/joint-vcf/AoU_srWGS_SV_PhaseI.vcf.gz .
    !gsutil -m cp {AOU_LR_DATA}/aou-wgs-sv/phase1/joint-vcf/AoU_srWGS_SV_PhaseI.vcf.gz.tbi .

In [ ]:
sr_sv_samples = !zgrep -m1 '^#CHROM' AoU_srWGS_SV_PhaseI.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(sr_sv_samples)

In [ ]:
# Long-read joint SV callsets at two calibration sensitivities (sens_09 and the stricter sens_07), and the
# sample IDs of sens_09.
if not os.path.exists("concat_annotated.sens_09.vcf.gz"):
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_09.vcf.gz .
        
if not os.path.exists("concat_annotated.sens_07.vcf.gz"):        
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_07.vcf.gz .

In [ ]:
lr_sv_samples = !zgrep -m1 '^#CHROM' concat_annotated.sens_09.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(lr_sv_samples)

In [ ]:
# Drop the HPRC samples (IDs starting HG or NA), leaving the 1,027 AoU long-read samples; then the samples
# that also have short-read SV calls.
common_samples_1027 = [element for element in lr_sv_samples if not (element.startswith('HG') or element.startswith('NA'))]
len(common_samples_1027)

In [ ]:
common_samples_989 = list(set(sr_sv_samples) & set(lr_sv_samples))
len(common_samples_989)

In [ ]:
# Short-read SV counts by type (the ALT allele with angle brackets and any ':' subtype removed).
!zcat AoU_srWGS_SV_PhaseI.vcf.gz | grep -v '^#' | cut -f5 | sed 's/:.*//g' | sed 's/[<>]//g' | sort | uniq -c

In [ ]:
# Long-read SV counts by SVTYPE (from the INFO field).
!zcat concat_annotated.sens_09.vcf.gz | grep -v '^#' | cut -f8 | sed 's/.\+SVTYPE=//' | sed 's/;.\+//' | sort | uniq -c

In [ ]:
!zcat concat_annotated.sens_09.vcf.gz | grep -v '^#' | cut -f8 | head

In [ ]:
# Switch conda to the faster libmamba solver.
!conda update -n base conda -y
!conda install -n base conda-libmamba-solver -y
!conda config --set solver libmamba -y